# TP3 — <!-- BEGIN SOLUTION -->SOLUTION (instructors only) — <!-- END SOLUTION -->Ocean model outputs: the CROCO regional model

**Suggested duration: ~3h** · **Prerequisites**: Course 3 (what is a numerical model), TP1 and TP2.

In TP1 and TP2 you worked with **observations** and with a **reanalysis** (GLORYS12: a global model constrained by observations). Today you open the raw output of a **regional ocean model**, CROCO, run over the North-East Tropical Pacific at ~5 km resolution. You will see that a model file does not look like an observation file — and learn how to turn it into maps and sections.

## Learning objectives
By the end of this TP you should be able to:
1. Explain what is different in a model output file: curvilinear grid, C-grid (rho/u/v points), terrain-following vertical levels, time counted from a model origin.
2. Map 2D surface fields: temperature, sea level, currents, wind stress.
3. Compute the **depth** of the model levels, then plot a **horizontal slice** at a fixed depth and a **vertical section**.
4. Compare the model with observations (OSTIA) and with the GLORYS12 reanalysis, on the same days.
5. Find and fix typical bugs when handling model outputs.

## How to work through this notebook
- Cells marked `# TODO` contain code **you need to complete** (replace the `...`). Everything else is already written.
- After each exercise, a **✅ Check** box tells you what a correct result should look like — use it to self-correct before moving on.
- Stuck on a `# TODO`? Below each exercise, a **💡 Hint** box (and, as a last resort, a **🔑 Solution** box) can be opened with a click. Try for a few minutes before opening them!
- **🚀 Advanced track** blocks are optional: do them only if you finished the main exercise.

---
## 0. Setup — imports, paths and helper functions

Run this cell as is. Besides the paths, it loads **`tools.py`** (file next to this notebook, imported as `to`): a small toolbox of functions commonly used with CROCO/ROMS outputs. They work on **numpy arrays** (use `.values` to get one from xarray). Today we use:

| Function | What it does |
|---|---|
| `to.zlevs(h, zeta, hc, Cs_r, Cs_w)` | depth (m, negative) of every model level → `z_r` (rho-levels), `z_w` (level interfaces) |
| `to.vinterp(var, z_r, depth, topo=h, cubic=1)` | interpolates a 3D field at a constant depth (m, negative) |
| `to.u2rho(u)`, `to.v2rho(v)` | move a field from u- or v-points to the rho-points (C-grid, see section 1) |
| `to.vort(u, v, pm, pn)`, `to.psi2rho(var)` | relative vorticity (on psi-points), and psi → rho points |

The cell also defines:
- `open_croco(avg_file, grd_file)`: opens a CROCO output file with xarray and fixes it (section 1 explains why);
- `base_map(ax, title, extent=None)`: adds land, coastlines and a grid to a map.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean

import tools as to                      # CROCO/ROMS toolbox (tools.py, next to this notebook)

# Data already on Kabre (read-only)
OBS_DIR = "/data0/user/gcambon/DATA/DATASETS_PSF"
if not Path(OBS_DIR).exists():          # not on Kabre: instructor's local copy
    OBS_DIR = "../data/DATASETS_PSF"

AVG_FILE = f"{OBS_DIR}/MODEL_DS/croco_avg_Y2023M12_light.nc"    # model outputs (daily averages)
GRD_FILE = f"{OBS_DIR}/MODEL_DS/croco_grd.nc"                   # model grid
OSTIA_FILE = f"{OBS_DIR}/COSTARICA_DS/sst_costarica_2023.nc"     # satellite SST (TP1)
GLORYS_FILE = f"{OBS_DIR}/COSTARICA_DS/3Docean_costarica_daily_2023_12.nc"   # reanalysis (TP2)

# Comparison box: Pacific side of Costa Rica only (the Caribbean is not in the CROCO domain)
LON_MIN, LON_MAX = -92.0, -85.0
LAT_MIN, LAT_MAX = 4.0, 12.0

CROCO_TIME_ORIGIN = "2000-01-01"   # model setting, NOT written in the file (see section 1)
FILL = 1e30                        # anything larger is a netCDF fill value, not data


def open_croco(avg_file, grd_file):
    """Open a CROCO output file and fix the three problems seen in section 1."""
    ds = xr.open_dataset(avg_file, decode_times=False, chunks={"time": 1})
    grd = xr.open_dataset(grd_file)
    # The grid file names the u/v dimensions differently from the output file
    grd = grd.rename({"eta_u": "eta_rho", "xi_v": "xi_rho"})
    # 1. Undeclared fill values (9.97e36) on land-only tiles -> NaN
    for name in ds.data_vars:
        if ds[name].dtype.kind == "f":
            ds[name] = ds[name].where(abs(ds[name]) < FILL)
    # 2. Clean grid (coordinates, bathymetry, masks, metrics) from the grid file
    for name in ["lon_rho", "lat_rho", "lon_u", "lat_u", "lon_v", "lat_v",
                 "h", "mask_rho", "mask_u", "mask_v", "pm", "pn", "f", "angle"]:
        ds[name] = grd[name]
    ds = ds.set_coords(["lon_rho", "lat_rho", "lon_u", "lat_u", "lon_v", "lat_v"])
    # Land points -> NaN (each variable with the mask of its own grid points)
    for name in ds.data_vars:
        dims = set(ds[name].dims)
        if {"eta_rho", "xi_rho"} <= dims and not name.startswith("mask"):
            ds[name] = ds[name].where(ds.mask_rho == 1)
        elif {"eta_rho", "xi_u"} <= dims and not name.startswith("mask"):
            ds[name] = ds[name].where(ds.mask_u == 1)
        elif {"eta_v", "xi_rho"} <= dims and not name.startswith("mask"):
            ds[name] = ds[name].where(ds.mask_v == 1)
    # 3. Time: seconds since the model origin -> real dates
    ds["time"] = np.datetime64(CROCO_TIME_ORIGIN) + ds.time.values.astype("timedelta64[s]")
    return ds


def base_map(ax, title, extent=None):
    """Land, coastlines and grid on a cartopy map (whole model domain by default)."""
    ax.add_feature(cfeature.LAND, facecolor="#d4c9a8", zorder=2)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.7, zorder=3)
    ax.gridlines(draw_labels=True, linewidth=0.3, linestyle="--", color="gray")
    if extent is not None:
        ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.set_title(title)


!ls -lh {OBS_DIR}/MODEL_DS

**✅ Check**: the `ls` shows `croco_avg_Y2023M12_light.nc` (~720 MB) and `croco_grd.nc` (~40 MB).

---
## 1. Anatomy of a model output file

Let's first open the model file **the usual way**, as we did for observations, and look at it.

In [ ]:
ds_raw = xr.open_dataset(AVG_FILE, decode_times=False)
ds_raw

**🎯 Your turn**: answer these questions from the output above (click on the 📄 and 🛢 icons to see attributes and values). Complete the cell to check your answers.
1. What are the dimensions of `temp`? Is there a `depth` dimension?
2. What are the dimensions of `lon_rho`? Of `u`? Of `v`?
3. What are the units of `time`? What is the first value?
4. What are the minimum and maximum of `lon_rho` and of `h` (bathymetry)?

In [ ]:
print("temp dims   :", ds_raw["temp"].dims)
print("lon_rho dims:", ds_raw["lon_rho"].dims)
### STUDENT: print("u dims      :", ...)        # TODO
### STUDENT: print("v dims      :", ...)        # TODO
### STUDENT: print("time units  :", ..., "| first value:", ...)   # TODO
### STUDENT: print("lon_rho min / max:", ..., ...)                # TODO
### STUDENT: print("h min / max      :", ..., ...)                # TODO
### BEGIN SOLUTION
print("u dims      :", ds_raw["u"].dims)
print("v dims      :", ds_raw["v"].dims)
print("time units  :", ds_raw["time"].attrs["units"], "| first value:", float(ds_raw["time"][0]))
print("lon_rho min / max:", float(ds_raw["lon_rho"].min()), float(ds_raw["lon_rho"].max()))
print("h min / max      :", float(ds_raw["h"].min()), float(ds_raw["h"].max()))
### END SOLUTION
### HINT: `ds_raw["u"].dims`, `ds_raw["time"].attrs["units"]`, `float(ds_raw["time"][0])`.
### HINT: `float(ds_raw["lon_rho"].min())` and `.max()`.

**✅ Check** — and what it means:

1. `temp` is `(time, s_rho, eta_rho, xi_rho)`: **no depth dimension**. CROCO uses **terrain-following levels** (`s_rho`, 50 levels): they follow the sea floor, so level 10 is at 30 m on the shelf and at 1500 m offshore. The depth of each level has to be **computed** (section 3). Also note: **level 0 is at the bottom**, the last level (`s_rho=-1`) is at the surface.
2. `lon_rho` is **2D**, `(eta_rho, xi_rho)`: in general a model grid is **curvilinear**, each grid point has its own longitude and latitude. `u` is on `(eta_rho, xi_u)` and `v` on `(eta_v, xi_rho)`: they are not on the same points as `temp`! This is the **Arakawa C-grid**:
   ```
        v           v
     u  ρ  u  ρ  u        ρ (rho): temp, salt, zeta, …   (551 × 350 points)
        v           v     u: eastward velocity, between two ρ in x   (550 × 350)
     u  ρ  u  ρ  u        v: northward velocity, between two ρ in y  (551 × 349)
        v           v
   ```
   To combine `u`, `v` and `temp` (e.g. a current speed, or arrows on a temperature map), `u` and `v` must first be moved to the ρ-points: that's `to.u2rho()` / `to.v2rho()`.
3. `time` is in **seconds**, without a reference date (`units = "second"`): **754 747 500 s** since… the model origin, a setting of the simulation (here 1 January 2000) that is **not written in the file**. That's a classic trap with model outputs: always ask the modeller!
4. `lon_rho` goes up to **9.97e36**, and so does `h`: these are **fill values** (the default netCDF "no data" value). The model was run in parallel on tiles, and tiles fully on land were not written. Since the file does not declare this fill value, xarray treats it as real data — a map or a mean would be meaningless.

👉 `open_croco()` fixes 3 and 4 (real dates, fill values → NaN), takes the clean grid from `croco_grd.nc`, and masks land points. From now on we use it.

The functions of `tools.py` work on **numpy arrays**, so we also extract the grid once as numpy arrays (`.values`).

In [ ]:
ds = open_croco(AVG_FILE, GRD_FILE)
print(ds["time"].values)
print("lon_rho:", float(ds.lon_rho.min()), "to", float(ds.lon_rho.max()))
print("lat_rho:", float(ds.lat_rho.min()), "to", float(ds.lat_rho.max()))
print("Grid spacing: %.1f to %.1f km" % (float(1 / ds.pm.max()) / 1e3, float(1 / ds.pm.min()) / 1e3))

# Grid as numpy arrays, for the tools.py functions
lon, lat = ds.lon_rho.values, ds.lat_rho.values        # (eta_rho, xi_rho)
h = ds.h.values                                        # bathymetry (m, positive), NaN on land
pm, pn, f = ds.pm.values, ds.pn.values, ds.f.values    # 1/dx, 1/dy (1/m), Coriolis (1/s)
hc, Cs_r, Cs_w = float(ds.hc), ds.Cs_rho.values, ds.Cs_w.values   # vertical grid parameters

**✅ Check**: 3 daily averages, **1, 2 and 3 December 2023** (at noon, the middle of each day). The domain covers **104.5°W–77°W, 2.3°N–19.4°N** (from southern Mexico to Colombia), with a grid spacing of **~5.2 to 5.6 km**.

💬 Compare with the products of TP1/TP2: OSTIA (0.05° ≈ 5.5 km), GLORYS12 (1/12° ≈ 9 km), WOA (1° ≈ 110 km).

---
# Part A — 2D surface fields

## A.1 Sea surface temperature

**🎯 Your turn**: select the **surface level** of `temp` and average it over the 3 days, then map it. Remember: the surface is the **last** level.

In [ ]:
### STUDENT: # TODO: surface level (hint: .isel(s_rho=...)), then mean over time
### STUDENT: sst_model = ...
### BEGIN SOLUTION
sst_model = ds["temp"].isel(s_rho=-1).mean(dim="time")
### END SOLUTION

fig, ax = plt.subplots(figsize=(11, 7), subplot_kw={"projection": ccrs.PlateCarree()})
sst_model.plot(ax=ax, x="lon_rho", y="lat_rho", transform=ccrs.PlateCarree(),
               cmap=cmocean.cm.thermal, cbar_kwargs={"label": "SST (°C)", "shrink": 0.7})
base_map(ax, "CROCO — SST, mean 1-3 December 2023")
plt.show()
### HINT: The surface is the last level: `.isel(s_rho=-1)`.
### HINT: Then `.mean(dim="time")`.

**✅ Check**: SST between **~26 °C and ~30.5 °C**. Note the **cold water** off the gulfs of **Tehuantepec** (Mexico, ~95°W) and **Papagayo** (Costa Rica/Nicaragua, ~87°W): in December, strong wind jets blow through gaps in the mountains, mix the ocean and bring up cold water (you saw it in the OSTIA time series in TP1). Note also the `x="lon_rho", y="lat_rho"` in `.plot()`: with 2D coordinates, xarray must be told which ones to use.

## A.2 Sea surface height

`zeta` is the sea surface height (m) relative to the model's reference level.

**🎯 Your turn**: map the 3-day mean of `zeta`, with a diverging colormap centred on its mean value.

In [ ]:
### STUDENT: # TODO: mean of zeta over time
### STUDENT: ssh_model = ...
### BEGIN SOLUTION
ssh_model = ds["zeta"].mean(dim="time")
### END SOLUTION

fig, ax = plt.subplots(figsize=(11, 7), subplot_kw={"projection": ccrs.PlateCarree()})
ssh_model.plot(ax=ax, x="lon_rho", y="lat_rho", transform=ccrs.PlateCarree(),
               cmap=cmocean.cm.balance, center=float(ssh_model.mean()),
               cbar_kwargs={"label": "SSH (m)", "shrink": 0.7})
base_map(ax, "CROCO — sea surface height, mean 1-3 December 2023")
plt.show()
### HINT: Same recipe as A.1, but `zeta` is already 2D (no level to select).

**✅ Check**: values between **~0.2 m and ~0.75 m**. The strongest feature is a round **high (~0.75 m) near 13°N, 100°W**: a large **anticyclonic eddy** (it is the warm swirl of A.1), shed by the Tehuantepec wind jet. **Lows** (blue) line up along 7-9°N, one of them centred near **9°N, 89°W**: the **Costa Rica Dome** — where the thermocline is shallow (TP1, WOA section), the sea surface is lower. The other round highs and lows of 50-200 km are **eddies** too.

## A.3 Surface currents

**🎯 Your turn**: take the **surface level** of `u` and `v` (3-day mean) as numpy arrays, move them to the ρ-points with `to.u2rho()` / `to.v2rho()`, then compute the current speed. Print the shapes before and after: that's the C-grid of section 1.

In [ ]:
### STUDENT: # TODO: surface level + time mean of u and v, as numpy arrays (.values)
### STUDENT: u_surf = ...
### STUDENT: v_surf = ...
### STUDENT:
### STUDENT: # TODO: move them to the rho-points with tools.py
### STUDENT: u_rho = ...
### STUDENT: v_rho = ...
### STUDENT:
### STUDENT: # TODO: current speed (m/s)
### STUDENT: speed = ...
### BEGIN SOLUTION
u_surf = ds["u"].isel(s_rho=-1).mean(dim="time").values
v_surf = ds["v"].isel(s_rho=-1).mean(dim="time").values

u_rho = to.u2rho(u_surf)
v_rho = to.v2rho(v_surf)

speed = np.hypot(u_rho, v_rho)
### END SOLUTION
print("u:", u_surf.shape, "->", u_rho.shape, "| v:", v_surf.shape, "->", v_rho.shape, "| temp:", lon.shape)

step = 15                                    # one arrow every 15 grid points (~80 km)
fig, ax = plt.subplots(figsize=(11, 7), subplot_kw={"projection": ccrs.PlateCarree()})
pc = ax.pcolormesh(lon, lat, speed, transform=ccrs.PlateCarree(), cmap=cmocean.cm.speed, vmax=1)
ax.quiver(lon[::step, ::step], lat[::step, ::step], u_rho[::step, ::step], v_rho[::step, ::step],
          transform=ccrs.PlateCarree(), scale=12, width=0.002, zorder=4)
fig.colorbar(pc, ax=ax, label="Current speed (m/s)", shrink=0.7)
base_map(ax, "CROCO — surface currents, mean 1-3 December 2023")
plt.show()
### HINT: Same selection as A.1 for `ds["u"]` and `ds["v"]`, then `.values` to get numpy arrays.
### HINT: `to.u2rho(u_surf)`, `to.v2rho(v_surf)`; speed = √(u² + v²): `np.hypot(u_rho, v_rho)`.

**✅ Check**: the shapes go from `(350, 550)` (u) and `(349, 551)` (v) to `(350, 551)`, the shape of the ρ-grid. Currents mostly below 0.5 m/s, with **~1 m/s** around the big eddy at 13°N, 100°W (turning **clockwise** around the SSH high: geostrophy in the Northern Hemisphere), off Papagayo, and in an **eastward current at 5-7°N** (the North Equatorial Countercurrent).

⚠️ If you get `ValueError: operands could not be broadcast together with shapes (350,550) (349,551)`, you tried to combine `u` and `v` before moving them to the ρ-points. (Here the grid is aligned with east/north — `angle` ≈ 0 — so no rotation of the vectors is needed; on a rotated grid it would be.)

## A.4 Wind stress: the gap winds

The model is forced at the surface by the wind, through the **wind stress** (N/m²): `sustr` (eastward, on u-points) and `svstr` (northward, on v-points).

**🎯 Your turn**: map the magnitude of the wind stress (3-day mean) and its direction.

In [ ]:
### STUDENT: # TODO: time mean of sustr and svstr (numpy), moved to the rho-points
### STUDENT: taux = ...
### STUDENT: tauy = ...
### STUDENT: tau = ...    # magnitude
### BEGIN SOLUTION
taux = to.u2rho(ds["sustr"].mean(dim="time").values)
tauy = to.v2rho(ds["svstr"].mean(dim="time").values)
tau = np.hypot(taux, tauy)
### END SOLUTION

step = 15
fig, ax = plt.subplots(figsize=(11, 7), subplot_kw={"projection": ccrs.PlateCarree()})
pc = ax.pcolormesh(lon, lat, tau, transform=ccrs.PlateCarree(), cmap=cmocean.cm.amp)
ax.quiver(lon[::step, ::step], lat[::step, ::step], taux[::step, ::step], tauy[::step, ::step],
          transform=ccrs.PlateCarree(), scale=2, width=0.002, zorder=4)
fig.colorbar(pc, ax=ax, label="Wind stress (N/m²)", shrink=0.7)
base_map(ax, "CROCO forcing — wind stress, mean 1-3 December 2023")
plt.show()
### HINT: `sustr` is on u-points, `svstr` on v-points: same treatment as `u` and `v` in A.3 (but they are already 2D, no level to select).

**✅ Check**: on these 3 days the dominant feature is the **Papagayo jet** (up to **~0.12 N/m²**), blowing offshore from the coast of Nicaragua/Costa Rica (~87°W, 11°N) and extending westward far into the Pacific along 10-12°N. Weaker jets blow through the gaps of **Tehuantepec** (~95°W, ~0.06 N/m²) and **Panama** (~80°W). South of 5°N: the southerly trade winds. Compare with the cold spots of A.1: wind jets → mixing and upwelling → cold surface water.

### 🚀 Advanced track A — eddies and vorticity

The relative vorticity ζ = ∂v/∂x − ∂u/∂y measures how much the flow rotates. Divided by the Coriolis parameter `f`, it gives the **Rossby number** ζ/f: |ζ/f| close to 1 means strong, small-scale eddies.

Compute ζ/f for the surface currents of A.3 with `to.vort(u, v, pm, pn)` — it takes `u` and `v` **on their own grid points** (not on the ρ-points!) and returns ζ on the **psi-points** (cell corners): move it to the ρ-points with `to.psi2rho()`. Map it with a diverging colormap (range −1 to 1). Do the eddies of A.2 appear? Which way do they turn?

In [ ]:
### STUDENT: # 🚀 Your code here
### BEGIN SOLUTION
vrt = to.vort(u_surf, v_surf, pm, pn)          # on psi-points: (349, 550)
rossby = to.psi2rho(vrt) / f                   # on rho-points: (350, 551)

fig, ax = plt.subplots(figsize=(11, 7), subplot_kw={"projection": ccrs.PlateCarree()})
pc = ax.pcolormesh(lon, lat, rossby, transform=ccrs.PlateCarree(), cmap=cmocean.cm.curl, vmin=-1, vmax=1)
fig.colorbar(pc, ax=ax, label="ζ / f", shrink=0.7)
base_map(ax, "CROCO — surface relative vorticity / f, mean 1-3 December 2023")
plt.show()

# Positive ζ/f = cyclonic (anticlockwise in the NH) = SSH lows; negative = anticyclonic = SSH highs.
# The strongest values (|ζ/f| ~ 1) are at the edges of the Tehuantepec and Papagayo jets.
### END SOLUTION
### HINT: `vrt = to.vort(u_surf, v_surf, pm, pn)` (u_surf, v_surf from A.3, before `u2rho`).
### HINT: `to.psi2rho(vrt) / f`, then `ax.pcolormesh(lon, lat, ..., cmap=cmocean.cm.curl, vmin=-1, vmax=1)`.

---
# Part B — 3D fields: depth of the levels, slices and sections

## B.1 Where are the model levels?

`to.zlevs(h, zeta, hc, Cs_r, Cs_w)` computes the depth (m, negative downwards) of every level from the bathymetry `h`, the sea level `zeta` and the vertical grid parameters stored in the file (`hc`, `Cs_rho`, `Cs_w`). It returns two arrays: `z_r` (depth of the ρ-levels, where `temp` is: 50 levels) and `z_w` (depth of the interfaces between levels: 51).

**🎯 Your turn**: compute the depths for the first day, then plot the **50 levels along the row at 9°N** (one line per level) together with the sea floor `-h`.

In [ ]:
### STUDENT: # TODO: sea level of the first day (numpy), then depths of all levels with tools.py
### STUDENT: zeta0 = ...
### STUDENT: z_r, z_w = ...
### BEGIN SOLUTION
zeta0 = ds["zeta"].isel(time=0).values
z_r, z_w = to.zlevs(h, zeta0, hc, Cs_r, Cs_w)
### END SOLUTION
print("z_r:", z_r.shape, "| z_w:", z_w.shape)

# Row of the grid closest to 9°N (rows have a constant latitude on this grid)
j9 = int(np.abs(lat[:, 0] - 9.0).argmin())
lon_row = lon[j9, :]
print(f"Row {j9}: latitude {lat[j9, 0]:.2f}°N")

fig, ax = plt.subplots(figsize=(12, 5))
### STUDENT: # TODO: one line per level: z_r along row j9 (hint: z_r[:, j9, :] is (level, x); ax.plot draws one line per column)
### STUDENT: ...
### BEGIN SOLUTION
ax.plot(lon_row, z_r[:, j9, :].T, color="0.3", linewidth=0.5)
### END SOLUTION
ax.plot(lon_row, -h[j9, :], color="k", linewidth=2, label="sea floor")
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Depth (m)")
ax.set_title("CROCO — the 50 terrain-following levels along 9°N")
ax.set_xlim(-105, -83)
ax.legend()
plt.show()

# Zoom on the upper 300 m
ax.set_ylim(-300, 5)
fig
### HINT: `ds["zeta"].isel(time=0).values`, then `to.zlevs(h, zeta0, hc, Cs_r, Cs_w)` (it returns two arrays).
### HINT: `z_r[:, j9, :]` is (level, longitude): `ax.plot(lon_row, z_r[:, j9, :].T)` draws one line per level.

**✅ Check**: `z_r` is `(50, 350, 551)` and `z_w` is `(51, 350, 551)`. The lines follow the sea floor (**terrain-following**): offshore (~3000-4000 m deep) the levels are spread over the whole water column, but **dense near the surface** (the zoom shows ~15 levels in the upper 100 m, spaced by a few metres at the top); on the steep slope off Costa Rica (~84°W), all 50 levels are squeezed into the shallower water column. The levels also wiggle at depth where the sea floor has seamounts. This is why there is no `depth` dimension: **the depth of a given level changes from one grid point to the next**.

## B.2 A vertical section across the Costa Rica Dome

**🎯 Your turn**: build the vertical section of temperature along 9°N, 3-day mean. For the depths, use the 3-day mean of `zeta`. On a section, each point has its own depth: we give `contourf` a 2D array of longitudes and a 2D array of depths.

In [ ]:
### STUDENT: # TODO: 3-day means (numpy) of temp (3D) and zeta (2D)
### STUDENT: temp_mean = ...
### STUDENT: zeta_mean = ...
### STUDENT: z_mean, _ = ...                # TODO: depths with tools.py
### STUDENT:
### STUDENT: # TODO: section along row j9: temperature and depth, (level, longitude)
### STUDENT: temp_sec = ...
### STUDENT: z_sec = ...
### BEGIN SOLUTION
temp_mean = ds["temp"].mean(dim="time").values
zeta_mean = ds["zeta"].mean(dim="time").values
z_mean, _ = to.zlevs(h, zeta_mean, hc, Cs_r, Cs_w)

temp_sec = temp_mean[:, j9, :]
z_sec = z_mean[:, j9, :]
### END SOLUTION
lon_sec = np.broadcast_to(lon_row, z_sec.shape)          # same longitudes on every level

fig, ax = plt.subplots(figsize=(12, 5))
cf = ax.contourf(lon_sec, z_sec, temp_sec, levels=np.arange(8, 31, 1), cmap=cmocean.cm.thermal, extend="both")
ax.contour(lon_sec, z_sec, temp_sec, levels=[20], colors="k", linewidths=2)
ax.fill_between(lon_row, -h[j9, :], -6000, color="#d4c9a8")   # sea floor
ax.set_ylim(-300, 0)
ax.set_xlim(-105, -83)
ax.set_facecolor("#d4c9a8")
fig.colorbar(cf, ax=ax, label="Temperature (°C)")
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Depth (m)")
ax.set_title("CROCO — temperature along 9°N, mean 1-3 December 2023 (black: 20 °C)")
plt.show()
### HINT: `ds["temp"].mean(dim="time").values` is (level, y, x): the section is `temp_mean[:, j9, :]`.
### HINT: Depths: `to.zlevs(h, zeta_mean, hc, Cs_r, Cs_w)`, then the same `[:, j9, :]`.

**✅ Check**: a warm surface layer (> 27 °C, ~30-40 m thick) above a sharp thermocline. The **20 °C isotherm** stays between **~35 and ~65 m**, with two domes where it rises to **~35 m**: near 100°W and near **88-89°W** — the **Costa Rica Dome**, just below the SSH low of A.2 — then it deepens again to ~65 m towards the coast. Same structure as the WOA climatology in TP1, but with much finer details (5 km vs 1°) and for 3 given days (not a long-term mean).

## B.3 A horizontal slice at 100 m

On an observation grid (WOA, ARMOR3D, GLORYS) you would write `.sel(depth=100)`. Here the levels are not at a fixed depth, so we must **interpolate vertically**, column by column: that's what `to.vinterp(var, z_r, depth, topo=h, cubic=1)` does (depth in m, **negative**; `topo=h` blanks the points where the sea floor is shallower than `depth`).

**🎯 Your turn**: map the temperature at **100 m** (3-day mean).

In [ ]:
### STUDENT: # TODO: temperature at 100 m, from temp_mean and z_mean (B.2)
### STUDENT: t100 = ...
### BEGIN SOLUTION
t100 = to.vinterp(temp_mean, z_mean, -100, topo=h, cubic=1)
### END SOLUTION

fig, ax = plt.subplots(figsize=(11, 7), subplot_kw={"projection": ccrs.PlateCarree()})
pc = ax.pcolormesh(lon, lat, t100, transform=ccrs.PlateCarree(), cmap=cmocean.cm.thermal)
ax.contour(lon, lat, h, levels=[100], colors="k", linewidths=0.5, transform=ccrs.PlateCarree())
fig.colorbar(pc, ax=ax, label="Temperature at 100 m (°C)", shrink=0.7)
base_map(ax, "CROCO — temperature at 100 m, mean 1-3 December 2023")
plt.show()
### HINT: `to.vinterp(temp_mean, z_mean, -100, topo=h, cubic=1)` — note the minus sign.
### HINT: (It prints "cubic interpolation": that's normal.)

**✅ Check**: much larger contrasts than at the surface (13 to 26 °C, vs 26 to 30.5 °C): **~13-15 °C along a band at 7-10°N** (the thermocline ridge, which includes the Costa Rica Dome), and **~25 °C in the core of the big eddy at 13°N, 100°W** (an anticyclone pushes the thermocline down). Areas shallower than 100 m (inside the thin black line, the 100 m isobath) are blank: there is no water at 100 m there.

⚠️ `to.vinterp(..., cubic=0)` (linear interpolation) fails with `ValueError: The truth value of an array ... is ambiguous` when `topo` is given: a small bug in this version of `tools.py`. Use `cubic=1`.

### 🚀 Advanced track B — the depth of the thermocline

Map the **depth of the 20 °C isotherm** (3-day mean), a common proxy for the thermocline depth. It's a vertical interpolation "in temperature space": for each column, find the two levels around 20 °C and interpolate their depths. Where is the thermocline shallowest?

In [ ]:
### STUDENT: # 🚀 Your code here
### BEGIN SOLUTION
T, Z = temp_mean, z_mean                             # (level, y, x), bottom -> surface
n = T.shape[0]
k = (T < 20).sum(axis=0)                             # levels colder than 20 °C (from the bottom)
inside = (k > 0) & (k < n)                           # 20 °C is crossed in this column
k1 = np.clip(k, 1, n - 1)[None]
take = lambda a, kk: np.take_along_axis(a, kk, axis=0)[0]
t0, t1, z0, z1 = take(T, k1 - 1), take(T, k1), take(Z, k1 - 1), take(Z, k1)
z20 = np.where(inside, z0 + (z1 - z0) * (20 - t0) / (t1 - t0), np.nan)

fig, ax = plt.subplots(figsize=(11, 7), subplot_kw={"projection": ccrs.PlateCarree()})
pc = ax.pcolormesh(lon, lat, -z20, transform=ccrs.PlateCarree(), cmap="viridis_r", vmin=0, vmax=120)
fig.colorbar(pc, ax=ax, label="Depth of the 20 °C isotherm (m)", shrink=0.7)
base_map(ax, "CROCO — depth of the 20 °C isotherm, mean 1-3 December 2023")
plt.show()

# Shallowest (~30-40 m) in the Costa Rica Dome (~9°N, 89°W), along the ridge at 7-9°N
# west of 95°W, and off Tehuantepec (~14°N, 95°W). Deepest (> 110 m) in the core of the
# anticyclonic eddy at 13°N, 100°W.
### END SOLUTION
### HINT: `k = (temp_mean < 20).sum(axis=0)` counts, in each column, the levels colder than 20 °C: 20 °C lies between level `k-1` and level `k`.
### HINT: `np.take_along_axis` picks, in each column, the value at a given level; then interpolate linearly.

---
# Part C — Model vs observations vs reanalysis

A model is **not** the truth: it has its own errors (resolution, forcing, parameterisations). CROCO here is a **free run** — unlike GLORYS12, it does not assimilate observations. Let's compare it, on **the same 3 days**, with:
- **OSTIA**: satellite SST (observation, TP1);
- **GLORYS12**: global reanalysis at 1/12° (model + data assimilation, TP2).

## C.1 Surface temperature: CROCO vs OSTIA

The comparison is made over a box on the **Pacific side of Costa Rica** (`LON_MIN…LAT_MAX`: 92°W–85°W, 4°N–12°N). Why not the whole Costa Rica box of TP2? Because the **Caribbean Sea is not part of the CROCO domain**: OSTIA would average water that CROCO does not have — an unfair comparison. Selecting a box with `.sel(longitude=slice(...))` works for OSTIA (1D coordinates), **not for CROCO** (2D coordinates): we use `.where(condition, drop=True)` instead.

**🎯 Your turn**: compute the box-mean SST of each day, for CROCO and for OSTIA, and put them in a table.

In [ ]:
DAYS = slice("2023-12-01", "2023-12-03")

# CROCO: surface level, then keep the points inside the box (2D coordinates -> .where)
croco_surf = ds["temp"].isel(s_rho=-1)
in_box = ((ds.lon_rho >= LON_MIN) & (ds.lon_rho <= LON_MAX)
          & (ds.lat_rho >= LAT_MIN) & (ds.lat_rho <= LAT_MAX))
### STUDENT: croco_box = ...          # TODO: keep the points of croco_surf inside the box
### STUDENT: croco_daily = ...        # TODO: mean over the box for each day (average over eta_rho and xi_rho)
### BEGIN SOLUTION
croco_box = croco_surf.where(in_box, drop=True)
croco_daily = croco_box.mean(dim=["eta_rho", "xi_rho"])
### END SOLUTION

# OSTIA: same days, same box (1D coordinates -> .sel), Kelvin -> Celsius
ds_ostia = xr.open_dataset(OSTIA_FILE)
### STUDENT: ostia_box = ...          # TODO: select DAYS and the box, convert to °C
### STUDENT: ostia_daily = ...        # TODO: mean over the box for each day
### BEGIN SOLUTION
ostia_box = ds_ostia["analysed_sst"].sel(time=DAYS, longitude=slice(LON_MIN, LON_MAX),
                                         latitude=slice(LAT_MIN, LAT_MAX)) - 273.15
ostia_daily = ostia_box.mean(dim=["latitude", "longitude"])
### END SOLUTION

table = pd.DataFrame({"CROCO (°C)": croco_daily.values, "OSTIA (°C)": ostia_daily.values},
                     index=pd.to_datetime(ostia_daily.time.values).strftime("%Y-%m-%d"))
table["CROCO - OSTIA"] = table["CROCO (°C)"] - table["OSTIA (°C)"]
print(table.round(2))
### HINT: CROCO: `croco_surf.where(in_box, drop=True)`, then `.mean(dim=["eta_rho", "xi_rho"])`.
### HINT: OSTIA: `.sel(time=DAYS, longitude=slice(LON_MIN, LON_MAX), latitude=slice(LAT_MIN, LAT_MAX))`, then `- 273.15`.

**✅ Check**: CROCO **~27.8 °C**, OSTIA **~28.8-28.9 °C**: the model is **~1 °C colder** than the satellite, every day.

💬 **Think**: is that difference an *error* of the model? Two things to keep in mind:
- OSTIA is a **foundation** SST (just below the daily warm skin), CROCO's top level is a ~1-3 m thick layer: they do not measure exactly the same thing.
- The Papagayo cold plume is **much stronger and wider** in the model (maps below): the model's wind forcing or mixing may be too strong these days.
- A 3-day comparison says little: tomorrow (TP4) you will compare over a whole seasonal cycle.

In [ ]:
# Side by side maps, same colour scale
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), subplot_kw={"projection": ccrs.PlateCarree()})
kw = dict(transform=ccrs.PlateCarree(), cmap=cmocean.cm.thermal, vmin=26, vmax=30.5, add_colorbar=False)
im = croco_box.mean(dim="time").plot(ax=axes[0], x="lon_rho", y="lat_rho", **kw)
ostia_box.mean(dim="time").plot(ax=axes[1], **kw)
extent = [LON_MIN, LON_MAX, LAT_MIN, LAT_MAX]
base_map(axes[0], "CROCO (free run, ~5 km)", extent)
base_map(axes[1], "OSTIA (satellite, 0.05°)", extent)
fig.colorbar(im, ax=axes, label="SST (°C), mean 1-3 December 2023", shrink=0.8)
plt.show()

**✅ Check**: both show the **cold Papagayo plume** off the coast at ~11°N, 87-89°W, but it is colder (~26.5 °C) and larger in CROCO than in OSTIA (~27.5 °C), and the rest of the box is ~1 °C cooler in the model. The Caribbean (top right) is blank in CROCO: outside the model domain. A free-running model reproduces the *processes* (jets, upwelling, eddies), not the exact state of a given day — that's what data assimilation (GLORYS) is for.

## C.2 Regional model vs global reanalysis: the Dome section

**🎯 Your turn**: build the same temperature section along 9°N from **GLORYS12** (3-day mean, `thetao`) and compare it with CROCO (B.2), over the same longitudes.

In [ ]:
ds_glo = xr.open_dataset(GLORYS_FILE)
### STUDENT: # TODO: GLORYS temperature: same 3 days, mean over time, latitude nearest to 9°N
### STUDENT: glo_sec = ...
### BEGIN SOLUTION
glo_sec = ds_glo["thetao"].sel(time=DAYS).mean(dim="time").sel(latitude=9.0, method="nearest")
### END SOLUTION

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
levels = np.arange(8, 31, 1)
# CROCO: section of B.2 (z negative)
cf = axes[0].contourf(lon_sec, z_sec, temp_sec, levels=levels, cmap=cmocean.cm.thermal, extend="both")
axes[0].contour(lon_sec, z_sec, temp_sec, levels=[20], colors="k", linewidths=2)
# GLORYS: depth is positive downwards -> use -depth to match CROCO
axes[1].contourf(glo_sec.longitude, -glo_sec.depth, glo_sec, levels=levels, cmap=cmocean.cm.thermal, extend="both")
axes[1].contour(glo_sec.longitude, -glo_sec.depth, glo_sec, levels=[20], colors="k", linewidths=2)
axes[0].set_ylim(-300, 0)
axes[0].set_ylabel("Depth (m)")
for ax in axes:
    ax.set_xlim(-92, -83.5)                # Pacific side, up to the coast
    ax.set_facecolor("#d4c9a8")
    ax.set_xlabel("Longitude (°E)")
axes[0].set_title("CROCO (regional, free run, ~5 km)")
axes[1].set_title("GLORYS12 (global reanalysis, 1/12°)")
fig.colorbar(cf, ax=axes, label="Temperature (°C), 9°N, 1-3 Dec 2023")
plt.show()
### HINT: `ds_glo["thetao"].sel(time=DAYS).mean(dim="time")`, then `.sel(latitude=9.0, method="nearest")`.

**✅ Check**: both show the thermocline rising around **88-89°W** (Costa Rica Dome) and deepening towards the coast, but the Dome is **sharper in CROCO** (20 °C isotherm from ~60 m at 92°W up to ~35 m at 88.5°W, then ~65 m at the coast) than in GLORYS (flatter, ~42-57 m).

⚠️ A convention to watch in a model/reanalysis comparison: CROCO depths are **negative** (z, counted upwards), GLORYS depths are **positive** (`depth`, counted downwards) — hence the `-glo_sec.depth` to put both on the same axis.

---
## D. 🐞 Debugging exercise

A colleague wants the map of the model's sea surface temperature on 1 December. *"It crashes — and once I got it to run, the map looked strange, with temperatures of 2 °C in the middle of the tropics!"* The cell contains **three bugs**: two raise an error, one is **silent**.

Run it, read the error message (the last line), fix, run again… until the map looks like A.1.

In [ ]:
### STUDENT: # 🐞 Fix me! (3 bugs)
### STUDENT: sst_bug = ds["temperature"].isel(time=0, s_rho=0)
### STUDENT:
### STUDENT: fig, ax = plt.subplots(figsize=(11, 7), subplot_kw={"projection": ccrs.PlateCarree()})
### STUDENT: sst_bug.plot(ax=ax, x="lon", y="lat", transform=ccrs.PlateCarree(), cmap=cmocean.cm.thermal)
### BEGIN SOLUTION
# Fixed version
sst_bug = ds["temp"].isel(time=0, s_rho=-1)

fig, ax = plt.subplots(figsize=(11, 7), subplot_kw={"projection": ccrs.PlateCarree()})
sst_bug.plot(ax=ax, x="lon_rho", y="lat_rho", transform=ccrs.PlateCarree(), cmap=cmocean.cm.thermal)
### END SOLUTION
base_map(ax, "CROCO — SST, 1 December 2023")
plt.show()
### BEGIN SOLUTION

# Bug 1: KeyError 'temperature'      -> the variable is called temp (list(ds.data_vars)).
# Bug 2: 'lon' / 'lat' do not exist -> the 2D coordinates are lon_rho / lat_rho.
# Bug 3 (silent): s_rho=0 is the BOTTOM level (1-4 °C in the deep ocean, warm on the
#        shelves where the bottom is shallow); the surface is s_rho=-1.
### END SOLUTION
### HINT: Read the last line of each error: check `list(ds.data_vars)` and `ds.coords`.
### HINT: A map at 2 °C in the tropics is not the surface… which level is `s_rho=0`? (section 1)

**✅ Check**: the fixed cell gives the same map as A.1 (26-30.5 °C). If your map shows **1-4 °C offshore and warm water only along the coasts**, you are still plotting the **bottom** level.

---
## E. In pairs: what does a model add?

Discuss with your partner and write 2-3 sentences for each question (you will reuse them on Friday):

1. Name **two things** you can get from CROCO that you could not get from the observations of TP1/TP2.
2. CROCO and OSTIA do not match exactly (C.1). Give **two reasons** that are not "the model is wrong".
3. For tomorrow's mini-project (TP4): to compare a model with observations **properly**, what should be the same in both (think: time, place, depth, grid)?

<!-- SOLUTION ONLY -->
### 👩‍🏫 Instructor notes

Expected elements of answer:
1. The **3D structure** everywhere (sections and slices at any depth, where Argo floats are sparse), **variables that are hard to observe** (vertical velocity `w`, mixing coefficient `AKt`, boundary layer depth `hbl`, heat fluxes), the **forcing** (wind stress) consistent with the ocean state, and the possibility to run **experiments** (e.g. without the wind jets) to understand *processes* — that's the main use of a regional model.
2. Different quantities (foundation SST vs a 1-3 m layer), different resolution/effective resolution, a free run has its own eddies (chaotic, not phase-locked to reality), errors in the atmospheric forcing, the satellite product itself is interpolated (L4) and has errors; 3 days is too short to conclude.
3. Same period and time averaging, same area (and the same grid: interpolate one onto the other), same depth/level, same variable definition and units; then look at statistics (bias, RMSE, correlation) over a long enough period (seasonal cycle).

Timing (~3h): section 1 ≈ 25 min (the anatomy discussion is the key part, do it with the whole group), Part A ≈ 50 min, Part B ≈ 50 min, Part C ≈ 30 min, D + E ≈ 25 min. If short on time: skip A.4 and C.2.

---
## To go further

- CROCO documentation (grid, vertical coordinates, outputs): <https://croco-ocean.gitlabpages.inria.fr/croco_doc/>
- `xgcm` (C-grid operations with xarray): <https://xgcm.readthedocs.io>
- Next: **TP4 mini-project** — choose a region and compare model, reanalysis and observations over a seasonal cycle.